# TUGAS MANDIRI PRAKTIKUM BIG DATA — PERTEMUAN 9
## Basis Data NoSQL: Pengelolaan Katalog Dokumen MongoDB & Integrasi PySpark

| Informasi | Detail |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Pertemuan** | 9 (Tugas Mandiri) |
| **Nama Mahasiswa** | Yarsya |
| **NPM** | 2505060066 |
| **Program Studi** | S1 Teknologi Informasi — Universitas Tidar |

---

### Skenario Tugas
Platform "Kopi Nusantara" mulai mendiversifikasi katalog menu mereka menggunakan database NoSQL berbasis dokumen (**MongoDB**). Karakteristik fleksibel MongoDB (*schema-less*) memungkinkan menu kategori *Kopi* menyimpan atribut spesifik seperti `tingkat_kepahitan`, sedangkan kategori *Makanan* menyimpan atribut `pedas_level` tanpa terikat skema kaku. Tugas ini mengimplementasikan operasi CRUD menyeluruh, kueri filter bertingkat, serta integrasi analitik ke dalam **Apache Spark DataFrame**.

## 0. Koneksi ke MongoDB & Inisialisasi Database
Menghubungkan klien PyMongo ke instans MongoDB lokal (port 27017) dan menyiapkan database `kopi_nusantara` serta collection `menu`.

In [1]:
import pymongo

# Membuat koneksi ke MongoDB lokal
client = pymongo.MongoClient("mongodb://localhost:27017/")

# Mengakses/membuat database kopi_nusantara dan collection menu
db = client["kopi_nusantara"]
koleksi_menu = db["menu"]

# Membersihkan data lama agar bersih dan reproducible
koleksi_menu.delete_many({})

print("Terhubung ke MongoDB!")
print("Database:", db.name)
print("Collection:", koleksi_menu.name)

Terhubung ke MongoDB!
Database: kopi_nusantara
Collection: menu


---
## Bagian A. Membangun Katalog Menu (Create — Bobot 20%)

Menyisipkan **14 dokumen menu** (minimal 12) dengan struktur field dinamis:
- Kategori **"Kopi"**: memiliki `nama_menu`, `harga`, `kategori`, dan `tingkat_kepahitan` (skala 1-5).
- Kategori **"Makanan"**: memiliki `nama_menu`, `harga`, `kategori`, dan `pedas_level` (skala 0-3).
- Field tambahan: `stok` dan `rekomendasi_barista`.

In [2]:
daftar_menu = [
    # Menu Kategori Kopi
    {"nama_menu": "Espresso Single", "kategori": "Kopi", "harga": 16000, "tingkat_kepahitan": 5, "stok": 40, "rekomendasi_barista": True},
    {"nama_menu": "Americano", "kategori": "Kopi", "harga": 20000, "tingkat_kepahitan": 4, "stok": 50, "rekomendasi_barista": False},
    {"nama_menu": "Kopi Susu Gula Aren", "kategori": "Kopi", "harga": 22000, "tingkat_kepahitan": 2, "stok": 60, "rekomendasi_barista": True},
    {"nama_menu": "Cappuccino", "kategori": "Kopi", "harga": 25000, "tingkat_kepahitan": 3, "stok": 35, "rekomendasi_barista": True},
    {"nama_menu": "Cold Brew Nusantara", "kategori": "Kopi", "harga": 28000, "tingkat_kepahitan": 4, "stok": 25, "rekomendasi_barista": True},
    {"nama_menu": "Caramel Macchiato", "kategori": "Kopi", "harga": 32000, "tingkat_kepahitan": 2, "stok": 30, "rekomendasi_barista": False},
    {"nama_menu": "Tubruk Robusta Temanggung", "kategori": "Kopi", "harga": 14000, "tingkat_kepahitan": 5, "stok": 45, "rekomendasi_barista": False},
    
    # Menu Kategori Makanan
    {"nama_menu": "Roti Bakar Cokelat Keju", "kategori": "Makanan", "harga": 18000, "pedas_level": 0, "stok": 30, "rekomendasi_barista": False},
    {"nama_menu": "Kentang Goreng Truffle", "kategori": "Makanan", "harga": 22000, "pedas_level": 0, "stok": 40, "rekomendasi_barista": True},
    {"nama_menu": "Cireng Bumbu Rujak Pedas", "kategori": "Makanan", "harga": 15000, "pedas_level": 2, "stok": 25, "rekomendasi_barista": False},
    {"nama_menu": "Tahu Gejrot Jeletot", "kategori": "Makanan", "harga": 17000, "pedas_level": 3, "stok": 20, "rekomendasi_barista": True},
    {"nama_menu": "Pisang Goreng Wijen Gula Palem", "kategori": "Makanan", "harga": 16000, "pedas_level": 0, "stok": 35, "rekomendasi_barista": False},
    {"nama_menu": "Mie Nyemek Pedas Spesial", "kategori": "Makanan", "harga": 24000, "pedas_level": 3, "stok": 30, "rekomendasi_barista": True},
    {"nama_menu": "Singkong Keju Merekah", "kategori": "Makanan", "harga": 15000, "pedas_level": 0, "stok": 25, "rekomendasi_barista": False}
]

hasil_insert = koleksi_menu.insert_many(daftar_menu)
print(f"Berhasil menyisipkan {len(hasil_insert.inserted_ids)} dokumen menu ke dalam MongoDB.")
print(f"Total dokumen di koleksi menu: {koleksi_menu.count_documents({})}.")

Berhasil menyisipkan 14 dokumen menu ke dalam MongoDB.
Total dokumen di koleksi menu: 14.


---
## Bagian B. Kueri Bertingkat (Read — Bobot 20%)

Menjalankan dan menampilkan hasil dari **tiga** kueri filter bertingkat:
1. Seluruh menu kategori **"Kopi"** dengan `tingkat_kepahitan` di atas 3.
2. Seluruh menu dengan `harga` di antara 15.000 dan 30.000 (inklusif).
3. Menu yang **tidak memiliki** field `pedas_level` sama sekali (menggunakan operator `$exists`).

In [3]:
# 1. Kueri 1: Menu Kopi dengan tingkat_kepahitan > 3
print("=== KUERI 1: Menu Kopi dengan tingkat_kepahitan > 3 ===")
kueri_1 = koleksi_menu.find(
    {"kategori": "Kopi", "tingkat_kepahitan": {"$gt": 3}},
    {"_id": 0, "nama_menu": 1, "kategori": 1, "harga": 1, "tingkat_kepahitan": 1}
)
for item in kueri_1:
    print(f"- {item['nama_menu']} | Harga: Rp {item['harga']:,} | Kepahitan: {item['tingkat_kepahitan']}/5")

=== KUERI 1: Menu Kopi dengan tingkat_kepahitan > 3 ===
- Espresso Single | Harga: Rp 16,000 | Kepahitan: 5/5
- Americano | Harga: Rp 20,000 | Kepahitan: 4/5
- Cold Brew Nusantara | Harga: Rp 28,000 | Kepahitan: 4/5
- Tubruk Robusta Temanggung | Harga: Rp 14,000 | Kepahitan: 5/5


In [4]:
# 2. Kueri 2: Menu dengan harga di antara 15.000 dan 30.000
print("=== KUERI 2: Menu dengan Harga antara Rp 15.000 dan Rp 30.000 ===")
kueri_2 = koleksi_menu.find(
    {"harga": {"$gte": 15000, "$lte": 30000}},
    {"_id": 0, "nama_menu": 1, "kategori": 1, "harga": 1}
).sort("harga", 1)

for item in kueri_2:
    print(f"- {item['nama_menu']:<32} | {item['kategori']:<8} | Rp {item['harga']:,}")

=== KUERI 2: Menu dengan Harga antara Rp 15.000 dan Rp 30.000 ===
- Cireng Bumbu Rujak Pedas         | Makanan  | Rp 15,000
- Singkong Keju Merekah            | Makanan  | Rp 15,000
- Espresso Single                  | Kopi     | Rp 16,000
- Pisang Goreng Wijen Gula Palem   | Makanan  | Rp 16,000
- Tahu Gejrot Jeletot              | Makanan  | Rp 17,000
- Roti Bakar Cokelat Keju          | Makanan  | Rp 18,000
- Americano                        | Kopi     | Rp 20,000
- Kopi Susu Gula Aren              | Kopi     | Rp 22,000
- Kentang Goreng Truffle           | Makanan  | Rp 22,000
- Mie Nyemek Pedas Spesial         | Makanan  | Rp 24,000
- Cappuccino                       | Kopi     | Rp 25,000
- Cold Brew Nusantara              | Kopi     | Rp 28,000


In [5]:
# 3. Kueri 3: Menu yang TIDAK memiliki field pedas_level ($exists: False)
print("=== KUERI 3: Menu yang TIDAK memiliki field 'pedas_level' ===")
kueri_3 = koleksi_menu.find(
    {"pedas_level": {"$exists": False}},
    {"_id": 0, "nama_menu": 1, "kategori": 1, "harga": 1, "tingkat_kepahitan": 1}
)
for item in kueri_3:
    print(f"- {item['nama_menu']} ({item['kategori']}) | Harga: Rp {item['harga']:,}")

=== KUERI 3: Menu yang TIDAK memiliki field 'pedas_level' ===
- Espresso Single (Kopi) | Harga: Rp 16,000
- Americano (Kopi) | Harga: Rp 20,000
- Kopi Susu Gula Aren (Kopi) | Harga: Rp 22,000
- Cappuccino (Kopi) | Harga: Rp 25,000
- Cold Brew Nusantara (Kopi) | Harga: Rp 28,000
- Caramel Macchiato (Kopi) | Harga: Rp 32,000
- Tubruk Robusta Temanggung (Kopi) | Harga: Rp 14,000


---
## Bagian C. Pembaruan Data (Update — Bobot 15%)

1. Menambahkan field `status` bernilai `"Tersedia"` ke seluruh dokumen menggunakan `update_many()`.
2. Mengubah nilai `status` menjadi `"Habis"` khusus pada 2 menu pilihan menggunakan `update_one()`.

In [6]:
# 1. Menambahkan status 'Tersedia' ke semua dokumen dengan update_many()
res_many = koleksi_menu.update_many({}, {"$set": {"status": "Tersedia"}})
print(f"Jumlah dokumen yang berhasil diperbarui ke status 'Tersedia': {res_many.modified_count}")

# 2. Mengubah status menjadi 'Habis' khusus pada 2 menu pilihan dengan update_one()
res_one_1 = koleksi_menu.update_one({"nama_menu": "Americano"}, {"$set": {"status": "Habis"}})
res_one_2 = koleksi_menu.update_one({"nama_menu": "Tahu Gejrot Jeletot"}, {"$set": {"status": "Habis"}})

print(f"Update Americano: {res_one_1.modified_count} dokumen")
print(f"Update Tahu Gejrot Jeletot: {res_one_2.modified_count} dokumen")

# Verifikasi data yang berstatus 'Habis'
print("\nDaftar Menu dengan Status 'Habis':")
for menu_habis in koleksi_menu.find({"status": "Habis"}, {"_id": 0, "nama_menu": 1, "status": 1}):
    print(f"- {menu_habis['nama_menu']} -> Status: {menu_habis['status']}")

Jumlah dokumen yang berhasil diperbarui ke status 'Tersedia': 14
Update Americano: 1 dokumen
Update Tahu Gejrot Jeletot: 1 dokumen

Daftar Menu dengan Status 'Habis':
- Americano -> Status: Habis
- Tahu Gejrot Jeletot -> Status: Habis


---
## Bagian D. Integrasi ke Spark DataFrame (Bobot 25%)

Mengambil seluruh data dari koleksi `menu`, mengubahnya menjadi pandas DataFrame sebagai jembatan tipe data, lalu mengonversinya ke **Spark DataFrame**.

Menampilkan:
1. Skema DataFrame (`printSchema()`).
2. Analisis rata-rata `harga` dan jumlah menu per `kategori` menggunakan `groupBy`.

In [7]:
import os
import sys
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, count, round, col

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

spark = SparkSession.builder \
    .appName("Tugas9-MongoDB-Spark-Yarsya") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Langkah 1: Ekstrak seluruh dokumen dari MongoDB
dokumen_list = list(koleksi_menu.find({}))

# Langkah 2: Konversi ke pandas DataFrame dan standarisasi kolom _id ke string
df_pandas = pd.DataFrame(dokumen_list)
df_pandas["_id"] = df_pandas["_id"].astype(str)

# Langkah 3: Konversi ke PySpark DataFrame
df_spark_menu = spark.createDataFrame(df_pandas)

# 1. Tampilkan skema hasil konversi
print("=== 1. SKEMA SPARK DATAFRAME (HASIL KONVERSI MONGODB) ===")
df_spark_menu.printSchema()

print("=== SAMPEL 10 BARIS SPARK DATAFRAME ===")
df_spark_menu.select("nama_menu", "kategori", "harga", "tingkat_kepahitan", "pedas_level", "status").show(10)

26/11/19 18:00:44 WARN Utils: Your hostname, meil resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/11/19 18:00:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 18:00:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


=== 1. SKEMA SPARK DATAFRAME (HASIL KONVERSI MONGODB) ===
root
 |-- _id: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- tingkat_kepahitan: double (nullable = true)
 |-- stok: long (nullable = true)
 |-- rekomendasi_barista: boolean (nullable = true)
 |-- status: string (nullable = true)
 |-- pedas_level: double (nullable = true)

=== SAMPEL 10 BARIS SPARK DATAFRAME ===


+--------------------+--------+-----+-----------------+-----------+--------+
|           nama_menu|kategori|harga|tingkat_kepahitan|pedas_level|  status|
+--------------------+--------+-----+-----------------+-----------+--------+
|     Espresso Single|    Kopi|16000|              5.0|        NaN|Tersedia|
|           Americano|    Kopi|20000|              4.0|        NaN|   Habis|
| Kopi Susu Gula Aren|    Kopi|22000|              2.0|        NaN|Tersedia|
|          Cappuccino|    Kopi|25000|              3.0|        NaN|Tersedia|
| Cold Brew Nusantara|    Kopi|28000|              4.0|        NaN|Tersedia|
|   Caramel Macchiato|    Kopi|32000|              2.0|        NaN|Tersedia|
|Tubruk Robusta Te...|    Kopi|14000|              5.0|        NaN|Tersedia|
|Roti Bakar Cokela...| Makanan|18000|              NaN|        0.0|Tersedia|
|Kentang Goreng Tr...| Makanan|22000|              NaN|        0.0|Tersedia|
|Cireng Bumbu Ruja...| Makanan|15000|              NaN|        2.0|Tersedia|

In [8]:
# 2. Rata-rata harga dan jumlah produk per kategori menggunakan groupBy
print("=== 2. ANALISIS RATA-RATA HARGA DAN JUMLAH MENU PER KATEGORI ===")
df_ringkasan_kategori = df_spark_menu.groupBy("kategori") \
    .agg(
        round(avg("harga"), 2).alias("rata_rata_harga"),
        count("nama_menu").alias("jumlah_menu")
    ) \
    .orderBy(col("rata_rata_harga").desc())

df_ringkasan_kategori.show()

=== 2. ANALISIS RATA-RATA HARGA DAN JUMLAH MENU PER KATEGORI ===


+--------+---------------+-----------+
|kategori|rata_rata_harga|jumlah_menu|
+--------+---------------+-----------+
|    Kopi|       22428.57|          7|
| Makanan|       18142.86|          7|
+--------+---------------+-----------+



---
## Bagian E. Kesimpulan Konseptual (Bobot 20%)

**Pertanyaan:** *Apa tantangan yang anda temui saat mengonversi data MongoDB (dengan field yang tidak seragam antar dokumen) menjadi Spark DataFrame (yang mengharuskan skema tabel seragam)? Bagaimana anda mengatasinya?*

**Jawaban Reflektif:**

Tantangan mendasar yang dihadapi dalam proses integrasi ini berakar pada perbedaan paradigma arsitektur data antara **MongoDB (*NoSQL Document-oriented, Schema-less*)** dan **Apache Spark (*Relational Distributed DataFrame, Schema-on-Read / Fixed-Schema*)**:

1. **Tantangan Heterogenitas Struktur dan Ketidakseragaman Field (*Sparse Data*):**
   Pada MongoDB, setiap dokumen dapat memiliki struktur kunci (*field*) yang berbeda. Dokumen berkategori *Kopi* memiliki field `tingkat_kepahitan` namun tidak memiliki `pedas_level`, sedangkan dokumen berkategori *Makanan* terjadi sebaliknya. Ketika dikonversi langsung ke struktur tabel Spark yang membutuhkan kolom seragam pada seluruh baris data, muncul ketidaksesuaian struktur skema jika tidak didefinisikan secara hati-hati.

2. **Tantangan Tipe Data Spesifik BSON:**
   MongoDB secara *default* menggunakan tipe BSON khusus seperti `ObjectId` untuk field `_id`. Spark DataFrame tidak memiliki tipe data bawaan `ObjectId`, sehingga upaya konversi langsung dari objek BSON mentah dapat memicu `TypeError` atau kegagalan serialisasi Py4J.

3. **Strategi Pemecahan dan Solusi:**
   - **Penanganan Kolom Hilang (*Missing Fields*):** Melalui perantara pustaka pandas (`pd.DataFrame(list_dokumen)`), seluruh *keys* unik dari gabungan dokumen dikonsolidasikan menjadi kumpulan kolom menyeluruh (*schema union*). Dokumen yang tidak memiliki field tertentu secara otomatis dipetakan menjadi nilai `NaN`/`None` di pandas, yang kemudian diterjemahkan oleh Spark menjadi nilai `null` bertipe numerik terprediksi (`double`/`float`) tanpa merusak baris data lainnya.
   - **Normalisasi Tipe Data BSON:** Kolom khusus seperti `_id` terlebih dahulu dikonversi menjadi representasi teks murni menggunakan `.astype(str)` sebelum dibaca oleh Spark.
   - **Skalabilitas Masa Depan:** Pada lingkungan produksi skala besar (*enterprise*), tantangan ini diatasi dengan mendefinisikan skema eksplisit (*explicit `StructType`*) saat membaca data atau memanfaatkan *MongoDB Spark Connector* resmi yang secara otomatis memetakan dokumen JSON/BSON ke skema *nullable columns* secara terdistribusi.